# FOMC statements -- compile

Takes the scraped link table (fomc_docs_raw.csv), filters to Statement
(HTML) rows, fetches each one's RAW body text (no cleaning yet), and saves
one row per document. This is the "raw, scraped and compiled" dataset --
intentionally unprocessed, so postprocessing choices can be revisited later
without re-fetching anything.

In [1]:
import os, sys
import pandas as pd
sys.path.append(os.path.join(os.path.dirname(os.getcwd()), "Helpers"))
from fomc_helpers import filter_statement_html, fetch_statement_texts

BASE = os.path.dirname(os.getcwd())
RAW_LINKS_PATH = os.path.join(BASE, "Data Input", "Raw Input", "fomc_docs_raw.csv")
COMPILED_PATH = os.path.join(BASE, "Data Input", "Raw Input", "fomc_statements_compiled_raw.csv")

## Filter to Statement (HTML) rows

In [2]:
df_combined = pd.read_csv(RAW_LINKS_PATH, parse_dates=["date"])
df_statements_html = filter_statement_html(df_combined)
print(df_statements_html.shape)
df_statements_html.head()

(264, 10)


,year,date,doc_type_raw,doc_type,title_text,anchor_label,url,filetype,year_page,source
0,1994,1994-02-04,Statement,Statement,Statement,Statement,https://www.federalreserve.gov/fomc/19940204de...,html,https://www.federalreserve.gov/monetarypolicy/...,Historical
1,1994,1994-03-22,Statement,Statement,Statement,Statement,https://www.federalreserve.gov/fomc/19940322de...,html,https://www.federalreserve.gov/monetarypolicy/...,Historical
2,1994,1994-04-18,Statement,Statement,Statement,Statement,https://www.federalreserve.gov/fomc/19940418de...,html,https://www.federalreserve.gov/monetarypolicy/...,Historical
3,1994,1994-05-17,Statement,Statement,Statement,Statement,https://www.federalreserve.gov/fomc/19940517de...,html,https://www.federalreserve.gov/monetarypolicy/...,Historical
4,1994,1994-08-16,Statement,Statement,Statement,Statement,https://www.federalreserve.gov/fomc/19940816de...,html,https://www.federalreserve.gov/monetarypolicy/...,Historical


## Fetch raw text for every statement link (this takes a while)

In [3]:
compiled = fetch_statement_texts(df_statements_html, pause=0.2)
print(compiled.shape)
print(f"{compiled['error'].notna().sum()} fetch errors")
compiled.head()

(264, 16)
0 fetch errors


,year,date,doc_type_raw,doc_type,title_text,anchor_label,url,filetype,year_page,source,http_status,page_title,text,pub_date_from_url,word_count,error
0,1994,1994-02-04,Statement,Statement,Statement,Statement,https://www.federalreserve.gov/fomc/19940204de...,html,https://www.federalreserve.gov/monetarypolicy/...,Historical,200,FRB: Press Release -- FOMC statement -- Februa...,"Release Date: February 4, 1994\nFor immediate ...",None,120,None
1,1994,1994-03-22,Statement,Statement,Statement,Statement,https://www.federalreserve.gov/fomc/19940322de...,html,https://www.federalreserve.gov/monetarypolicy/...,Historical,200,FRB: Press Release -- FOMC statement -- March ...,"Release Date: March 22, 1994\nFor immediate re...",None,60,None
2,1994,1994-04-18,Statement,Statement,Statement,Statement,https://www.federalreserve.gov/fomc/19940418de...,html,https://www.federalreserve.gov/monetarypolicy/...,Historical,200,FRB: Press Release -- FOMC statement -- April ...,"Release Date: April 18, 1994\nFor immediate re...",None,57,None
3,1994,1994-05-17,Statement,Statement,Statement,Statement,https://www.federalreserve.gov/fomc/19940517de...,html,https://www.federalreserve.gov/monetarypolicy/...,Historical,200,FRB: Press Release -- FOMC statement -- May 17...,"Release Date: May 17, 1994\nFor immediate rele...",None,185,None
4,1994,1994-08-16,Statement,Statement,Statement,Statement,https://www.federalreserve.gov/fomc/19940816de...,html,https://www.federalreserve.gov/monetarypolicy/...,Historical,200,FRB: Press Release -- FOMC statement -- August...,"Release Date: August 16, 1994\nFor immediate r...",None,200,None


## Save the raw compiled dataset

In [4]:
out_cols = ["date", "year", "url", "doc_type", "page_title", "text", "word_count", "http_status", "error"]
compiled = compiled.rename(columns={"text": "raw_text"})
out_cols = [c if c != "text" else "raw_text" for c in out_cols]
compiled = compiled[out_cols]

os.makedirs(os.path.dirname(COMPILED_PATH), exist_ok=True)
compiled.to_csv(COMPILED_PATH, index=False)
print(f"Saved {len(compiled):,} rows to: {COMPILED_PATH}")

Saved 264 rows to: C:\Users\User\Desktop\DISSERTATION PAPERS\FED_TREASURY_DATA\Data Input\Raw Input\fomc_statements_compiled_raw.csv
